# Instalasi Library

In [ ]:
!pip install transformers datasets FreeSimpleGUI accelerate evaluate sacrebleu sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.6/733.6 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00


# Pengambilan Dataset OPUS

In [ ]:
import os
import requests
import tarfile

# 1. URL langsung ke file korpus moses dari OPUS untuk wikimedia id-su
url = "https://object.pouta.csc.fi/OPUS-wikimedia/v20230407/moses/id-su.txt.zip"
zip_path = "id-su.txt.zip"

print("Sedang mengunduh dataset dari OPUS...")
r = requests.get(url, stream=True)
with open(zip_path, 'wb') as f:
    for chunk in r.iter_content(chunk_size=1024):
        if chunk:
            f.write(chunk)

print("Mengunduh selesai! Sedang mengekstrak file...")
# Extract file zip nya
import zipfile
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset_opus")

print("File berhasil diekstrak ke folder 'dataset_opus'!")
print("Isi folder:", os.listdir("dataset_opus"))

Sedang mengunduh dataset dari OPUS...
Mengunduh selesai! Sedang mengekstrak file...
File berhasil diekstrak ke folder 'dataset_opus'!
Isi folder: ['wikimedia.id-su.id', 'wikimedia.id-su.xml', 'wikimedia.id-su.su', 'LICENSE', 'README']


# Membaca dan Menyusun Dataset

In [ ]:
import pandas as pd

# Path ke file hasil ekstrak (sesuaikan nama file jika ada perbedaan dari os.listdir tadi)
file_id = "dataset_opus/wikimedia.id-su.id"
file_su = "dataset_opus/wikimedia.id-su.su"

# Membaca file baris demi baris
with open(file_id, 'r', encoding='utf-8') as f:
    ind_lines = [line.strip() for line in f.readlines()]

with open(file_su, 'r', encoding='utf-8') as f:
    sun_lines = [line.strip() for line in f.readlines()]

# Satukan ke dalam DataFrame Pandas
df_clean = pd.DataFrame({
    'ind_Latn': ind_lines,
    'sun_Latn': sun_lines
})

# Tampilkan 5 baris pertama untuk memastikan keselarasan arti
print(f"Total baris data yang didapat: {len(df_clean)}")
print("\nContoh data:")
print(df_clean.head())

Total baris data yang didapat: 5427

Contoh data:
                                            ind_Latn  \
0  Congkrang adalah alat-alat pertanian yang digu...   
1                Artikel ini adalah sebuah rintisan.   
2  Anda dapat membantu Wikipedia dengan mengemban...   
3  Congkrang terbuat dari besi dengan gagang dari...   
4  Bandung: PT Kiblat Buku Utama. ↑ Sirat, Muhidd...   

                                            sun_Latn  
0  Congkrang nyaéta pakakas tatanén anu dipaké pi...  
1  Artikel ieu mangrupa taratas, perlu disampurna...  
2  Upami sadérék uninga langkung paos perkawis ie...  
3  Congkrang dijieunna tina beusi gagangna tina k...  
4  Bandung: PT Kiblat Buku Utama. ↑ Sirat, Muhidd...  


In [ ]:
# Menyimpan ke folder local di Google Colab sementara
df_clean.to_csv("dataset_translator_idsu.csv", index=False, encoding="utf-8")

print("File 'dataset_translator_idsu.csv' berhasil dibuat!")

File 'dataset_translator_idsu.csv' berhasil dibuat!


# Cleaning Dataset

In [ ]:
import pandas as pd
import re

# 1. Muat file CSV yang berisi data wikimedia tadi
df = pd.read_csv("dataset_translator_idsu.csv")

def clean_wikipedia_noise(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'\[\[([^\]|]+)\]\]', r'\1', text)
    text = re.sub(r'\[\[[^\]|]+\|([^\]]+)\]\]', r'\1', text)
    text = re.sub(r'\[\d+\]', '', text)

    # KASUS 3: Merapikan spasi ganda yang muncul akibat proses penghapusan di atas
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# 2. Terapkan fungsi pembersihan ke kedua kolom (Indonesia & Sunda)
df['ind_Latn'] = df['ind_Latn'].apply(clean_wikipedia_noise)
df['sun_Latn'] = df['sun_Latn'].apply(clean_wikipedia_noise)

# 3. Filter opsional: Buang baris yang rusak atau mengandung karakter sisa daftar pustaka '↑'
df = df[~df['ind_Latn'].str.contains('↑', na=False)]

# 4. Simpan menjadi file CSV bersih final
df.to_csv("dataset_translator_idsu_perfect.csv", index=False, encoding="utf-8")

print(f"Pembersihan Selesai!")
print(f"Jumlah baris data yang siap disuapkan ke NLLB-200: {len(df)} baris.")

Pembersihan Selesai!
Jumlah baris data yang siap disuapkan ke NLLB-200: 5277 baris.


# Load Dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Muat file CSV bersih
raw_dataset = load_dataset("csv", data_files="dataset_translator_idsu_perfect.csv")
# Sel Tambahan: Menyaring baris kosong (NaN / None)
raw_dataset = raw_dataset.filter(lambda x: isinstance(x['ind_Latn'], str) and isinstance(x['sun_Latn'], str))

print("Dataset berhasil dimuat!")
print(raw_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/5277 [00:00<?, ? examples/s]

Dataset berhasil dimuat!
DatasetDict({
    train: Dataset({
        features: ['ind_Latn', 'sun_Latn'],
        num_rows: 5272
    })
})


# Split Dataset

In [ ]:
# 1. Bagi data menjadi Train (90%) dan Sisa (10%)
split_dataset = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)

# 2. Bagi sisa data (10%) tadi menjadi Validation (5%) dan Test Akhir (5%)
val_test_split = split_dataset["test"].train_test_split(test_size=0.5, seed=42)

# 3. Satukan kembali ke dalam struktur dictionary yang rapi
final_splits = {
    "train": split_dataset["train"],
    "validation": val_test_split["train"],
    "test": val_test_split["test"]
}

print("=== Hasil Pembagian Dataset ===")
print(f"Jumlah Data Latihan (Train)       : {len(final_splits['train'])} baris")
print(f"Jumlah Data Pantau (Validation)   : {len(final_splits['validation'])} baris")
print(f"Jumlah Data Ujian Akhir (Test)    : {len(final_splits['test'])} baris")

=== Hasil Pembagian Dataset ===
Jumlah Data Latihan (Train)       : 4744 baris
Jumlah Data Pantau (Validation)   : 264 baris
Jumlah Data Ujian Akhir (Test)    : 264 baris


# Tokenizer NLLB

In [ ]:
model_checkpoint = "facebook/nllb-200-distilled-600M"

# Memuat tokenizer resmi NLLB-200
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer NLLB-200 berhasil diunduh dan siap digunakan!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

Tokenizer NLLB-200 berhasil diunduh dan siap digunakan!


# Preprocessing Dua Arah

In [ ]:
def preprocess_indo_to_sunda(examples):
    # Pastikan input dikonversi ke string murni dan membaca kolom 'ind_Latn' & 'sun_Latn'
    inputs = [str(text) for text in examples["ind_Latn"]]
    targets = [str(text) for text in examples["sun_Latn"]]

    # Set bahasa asal untuk tokenizer jika menggunakan NLLB
    tokenizer.src_lang = "ind_Latn"
    tokenizer.tgt_lang = "sun_Latn"

    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

def preprocess_sunda_to_indo(examples):
    # Sebaliknya, arah Sunda ke Indonesia
    inputs = [str(text) for text in examples["sun_Latn"]]
    targets = [str(text) for text in examples["ind_Latn"]]

    # Set bahasa asal untuk tokenizer jika menggunakan NLLB
    tokenizer.src_lang = "sun_Latn"
    tokenizer.tgt_lang = "ind_Latn"

    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

# Tokenisasi Dataset Dua Arah

In [ ]:
from datasets import concatenate_datasets

# 1. Jalankan tokenisasi untuk masing-masing arah menggunakan 'final_splits' milikmu
tokenized_train_idsu = final_splits["train"].map(
    preprocess_indo_to_sunda, batched=True
)
tokenized_val_idsu = final_splits["validation"].map(
    preprocess_indo_to_sunda, batched=True
)

tokenized_train_suid = final_splits["train"].map(
    preprocess_sunda_to_indo, batched=True
)
tokenized_val_suid = final_splits["validation"].map(
    preprocess_sunda_to_indo, batched=True
)

# 2. Gabungkan data dan acak (Shuffle) agar seimbang
train_dataset_dua_arah = concatenate_datasets(
    [tokenized_train_idsu, tokenized_train_suid]
).shuffle(seed=42)
val_dataset_dua_arah = concatenate_datasets(
    [tokenized_val_idsu, tokenized_val_suid]
).shuffle(seed=42)

print("=== Ukuran Dataset Dua Arah ===")
print(f"Data Latihan (Train) ganda : {len(train_dataset_dua_arah)} baris")
print(f"Data Validasi (Eval) ganda : {len(val_dataset_dua_arah)} baris")
print("✅ Tokenisasi dan penggabungan data berhasil tanpa error!")

Map:   0%|          | 0/4744 [00:00<?, ? examples/s]

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/4744 [00:00<?, ? examples/s]

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

=== Ukuran Dataset Dua Arah ===
Data Latihan (Train) ganda : 9488 baris
Data Validasi (Eval) ganda : 528 baris
✅ Tokenisasi dan penggabungan data berhasil tanpa error!


# Evaluasi BLEU

In [ ]:
import evaluate
import numpy as np
from transformers import DataCollatorForSeq2Seq

# 1. Inisialisasi Data Collator khusus untuk model Seq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_checkpoint)

# 2. Muat metrik evaluasi SacreBLEU untuk mengukur akurasi translasi
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Ganti -100 pada label karena itu adalah token padding yang diabaikan
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

print("Data Collator dan Fungsi Evaluasi BLEU siap!")

Data Collator dan Fungsi Evaluasi BLEU siap!


# Load Model NLLB

In [ ]:
from transformers import AutoModelForSeq2SeqLM

# Memuat arsitektur model Seq2Seq
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

print("Model NLLB-200 berhasil dimuat ke GPU!")

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model NLLB-200 berhasil dimuat ke GPU!


# Fine Tuning Model

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./hasil_riset_nllb_bidirectional",
    eval_strategy="epoch",            # Menggunakan versi transformers terbaru
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,               # Waktu pelatihan akan 2x lebih lama dari sebelumnya
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_dua_arah, # Diubah ke dataset gabungan baru
    eval_dataset=val_dataset_dua_arah,   # Diubah ke dataset gabungan baru
    processing_class=tokenizer,           # Parameter terbaru pengganti tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,      # Tetap menggunakan fungsi BLEU milikmu
)

print("Inisialisasi selesai. Menjalankan Fine-Tuning Dua Arah...")
trainer.train()

Inisialisasi selesai. Menjalankan Fine-Tuning Dua Arah...


Epoch,Training Loss,Validation Loss,Bleu
1,1.620611,1.548511,38.386345
2,1.525013,1.516699,39.425150
3,1.389077,1.512629,39.633798


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3558, training_loss=1.5307993725352638, metrics={'train_runtime': 2468.0704, 'train_samples_per_second': 11.533, 'train_steps_per_second': 1.442, 'total_flos': 4567917891158016.0, 'train_loss': 1.5307993725352638, 'epoch': 3.0})

# Simpan Model

In [ ]:
model_bidirectional_dir = "./translator_dua_arah_final"

trainer.save_model(model_bidirectional_dir)
tokenizer.save_pretrained(model_bidirectional_dir)

print(f"✅ Model dua arah berhasil disimpan di: {model_bidirectional_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model dua arah berhasil disimpan di: ./translator_dua_arah_final


# Evaluasi Akhir

In [ ]:
from datasets import concatenate_datasets

# 1. Tokenisasi data Test untuk kedua arah terlebih dahulu
tokenized_test_idsu = final_splits["test"].map(
    preprocess_indo_to_sunda, batched=True
)
tokenized_test_suid = final_splits["test"].map(
    preprocess_sunda_to_indo, batched=True
)

# 2. Gabungkan data test dari kedua arah dan acak (Shuffle)
test_dataset_dua_arah = concatenate_datasets(
    [tokenized_test_idsu, tokenized_test_suid]
).shuffle(seed=42)

print(f"Jumlah data Test gabungan dua arah: {len(test_dataset_dua_arah)} baris")

hasil_test = trainer.evaluate(eval_dataset=test_dataset_dua_arah)

print("\n=== HASIL UJIAN AKHIR (TEST SET DUA ARAH) ===")
print(f"Final BLEU Score : {hasil_test['eval_bleu']:.2f}")
print(f"Final Loss       : {hasil_test['eval_loss']:.4f}")

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Jumlah data Test gabungan dua arah: 528 baris


Training Loss,Validation Loss,Epoch,Bleu
1.389068,1.483370,3,38.689214



=== HASIL UJIAN AKHIR (TEST SET DUA ARAH) ===
Final BLEU Score : 38.69
Final Loss       : 1.4834


# Tampilan Translator


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Define the directory where the fine-tuned model and tokenizer are saved
MODEL_DIR = "/content/drive/MyDrive/translator_dua_arah_final"

# Load the model and tokenizer from the specified directory
model_uji = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to("cuda")
tokenizer_uji = AutoTokenizer.from_pretrained(MODEL_DIR)

# --- UI Components (Replicating User's Example Design) ---

# Header
display(HTML("""
<div style="background:linear-gradient(135deg,#1e3c72 0%,#2a5298 100%); padding:20px; border-radius:10px; color:white; text-align:center; font-family:Inter,Segoe UI,sans-serif; font-weight:700; margin-bottom:20px; box-shadow:0 4px 15px rgba(0,0,0,0.1);">
  <h1 style="margin:0; font-size:22px; font-weight:700;">Integrated Linguistic Tools</h1>
</div>
"""))

display(HTML('<p style="font-size:20px; font-weight:600; color:#2a5298; font-family:Inter,sans-serif; margin:0 0 15px 0;">NLLB-200 Fine-tuned Translator (Indonesian ↔ Sundanese)</p>'))

# Language Selection
LANGUAGES = ["Indonesia", "Sunda"]

w_source = widgets.Dropdown(
    options=LANGUAGES, value="Indonesia",
    description='',
    layout=widgets.Layout(width="100%")
)
w_target = widgets.Dropdown(
    options=LANGUAGES, value="Sunda",
    description='',
    layout=widgets.Layout(width="100%")
)
w_swap = widgets.Button(
    description='Swap',
    layout=widgets.Layout(width="56px", height="36px", margin="0 8px")
)

lbl_src = widgets.HTML('<span style="font-family:Inter,sans-serif; font-size:14px; color:#333; display:block; margin-bottom:4px;">Source Language</span>')
lbl_tgt = widgets.HTML('<span style="font-family:Inter,sans-serif; font-size:14px; color:#333; display:block; margin-bottom:4px;">Target Language</span>')
lbl_spacer = widgets.HTML('<span style="display:block; height:22px;"></span>')

col_src  = widgets.VBox([lbl_src, w_source], layout=widgets.Layout(flex='1'))
col_swap = widgets.VBox([lbl_spacer, w_swap], layout=widgets.Layout(align_items='center', justify_content='flex-end'))
col_tgt  = widgets.VBox([lbl_tgt, w_target], layout=widgets.Layout(flex='1'))

lang_row = widgets.HBox(
    [col_src, col_swap, col_tgt],
    layout=widgets.Layout(align_items="flex-start", margin="0 0 15px 0")
)
display(lang_row)

# Text Input Area
display(HTML('<p style="font-family:Inter,sans-serif; color:#2a5298; font-size:14px; font-weight:500; margin:15px 0 6px 0;">Enter Text to Translate</p>'))
w_input = widgets.Textarea(
    placeholder="Ketik kalimat di sini...",
    description='',
    layout=widgets.Layout(width="100%", height="150px")
)
display(w_input)

# Translate Button & Output Area
w_btn = widgets.Button(
    description="Translate",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="40px", margin="10px 0 0 0")
)
w_output = widgets.Output()

def swap_languages(b):
    src = w_source.value
    tgt = w_target.value
    w_source.value = tgt
    w_target.value = src

w_swap.on_click(swap_languages)

def on_translate(b):
    with w_output:
        clear_output(wait=True)
        text     = w_input.value.strip()
        src_lang = w_source.value
        tgt_lang = w_target.value

        if not text:
            display(HTML('<div style="color:#e67e22; padding:10px; background:#fff3cd; border-radius:8px;">Silakan masukkan teks terlebih dahulu.</div>'))
            return
        if src_lang == tgt_lang:
            display(HTML('<div style="color:#e74c3c; padding:10px; background:#fde8e8; border-radius:8px;">Bahasa asal dan tujuan tidak boleh sama.</div>'))
            return
        if not ((src_lang == "Indonesia" and tgt_lang == "Sunda") or (src_lang == "Sunda" and tgt_lang == "Indonesia")):
             display(HTML('<div style="color:#e74c3c; padding:10px; background:#fde8e8; border-radius:8px;">Model ini hanya mendukung terjemahan antara Indonesia dan Sunda.</div>'))
             return

        display(HTML('<div style="color:#2980b9; padding:8px;">Sedang menerjemahkan...</div>'))

        # Map user-friendly language names to internal model language codes
        lang_map = {
            "Indonesia": "ind_Latn",
            "Sunda": "sun_Latn",
        }
        src_lang_code = lang_map[src_lang]
        tgt_lang_code = lang_map[tgt_lang]

        tokenizer_uji.src_lang = src_lang_code
        inputs = tokenizer_uji(text, return_tensors="pt").to("cuda")

        forced_bos_token_id = tokenizer_uji.convert_tokens_to_ids(tgt_lang_code)
        outputs = model_uji.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=128
        )

        translated_text = tokenizer_uji.decode(outputs[0], skip_special_tokens=True)

        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#f8f9fa; padding:20px; border-radius:10px; border:1px solid #dee2e6; margin-top:8px; font-family:Inter,Segoe UI,sans-serif;">
          <p style="margin:0 0 8px 0; color:#2a5298; font-weight:600; font-size:13px;">Translated Text:</p>
          <p style="margin:0; font-size:17px; color:#1e3c72; font-weight:500;">{translated_text}</p>
        </div>
        """))

w_btn.on_click(on_translate)
display(w_btn)
display(w_output)


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Textarea(value='', layout=Layout(height='150px', width='100%'), placeholder='Ketik kalimat di sini...')

Button(button_style='primary', description='Translate', layout=Layout(height='40px', margin='10px 0 0 0', widt…

Output()

In [ ]:
from google.colab import drive

# Menghubungkan Google Drive
drive.mount('/content/drive')